In [2]:
from collections import Counter

words = {
    "hug": 2,
    "hugs": 1,
    "pug": 1
}

splits = {
    "hug": ["h", "##u", "##g"],
    "hugs": ["h", "##u", "##g", "##s"],
    "pug": ["p", "##u", "##g"]
}

print("Initial Splits:")
for word, tokens in splits.items():
    print(word, "->", tokens)

def calculate_token_frequency(splits, word_freq):

    token_freq = Counter()

    for word, freq in word_freq.items():

        tokens = splits[word]

        for token in tokens:
            token_freq[token] += freq

    return token_freq

def calculate_pair_frequency(splits, word_freq):

    pair_freq = Counter()

    for word, freq in word_freq.items():

        tokens = splits[word]

        for i in range(len(tokens) - 1):

            pair = (tokens[i], tokens[i + 1])

            pair_freq[pair] += freq

    return pair_freq

def calculate_scores(token_freq, pair_freq):

    scores = {}

    for pair, freq in pair_freq.items():

        first = pair[0]
        second = pair[1]

        score = freq / (
            token_freq[first] * token_freq[second]
        )

        scores[pair] = score

    return scores

def merge_pair(splits, pair):

    new_token = pair[0] + pair[1].replace("##", "")

    for word in splits:

        tokens = splits[word]

        new_tokens = []

        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1:

                current_pair = (
                    tokens[i],
                    tokens[i + 1]
                )

                if current_pair == pair:

                    new_tokens.append(new_token)

                    i += 2

                    continue

            new_tokens.append(tokens[i])

            i += 1

        splits[word] = new_tokens

    return new_token

vocab = {
    "h",
    "p",
    "##u",
    "##g",
    "##s"
}

print("\nInitial Vocabulary:")
print(vocab)

for step in range(3):

    token_freq = calculate_token_frequency(
        splits,
        words
    )

    pair_freq = calculate_pair_frequency(
        splits,
        words
    )

    scores = calculate_scores(
        token_freq,
        pair_freq
    )

    if not scores:
        break

    best_pair = max(
        scores,
        key=scores.get
    )

    new_token = merge_pair(
        splits,
        best_pair
    )

    vocab.add(new_token)

    print("\nStep", step + 1)

    print("Best Pair:", best_pair)

    print("Score:", scores[best_pair])

    print("New Token:", new_token)

    print("Vocabulary:", vocab)

    print("Splits:")

    for word, tokens in splits.items():
        print(word, "->", tokens)

def tokenize_word(word, vocab):

    tokens = []

    while len(word) > 0:

        found = False

        # Check longest possible subword first
        for i in range(len(word), 0, -1):

            part = word[:i]

            if len(tokens) > 0:
                part = "##" + part

            if part in vocab:

                tokens.append(part)

                word = word[i:]

                found = True

                break

        if not found:

            return ["[UNK]"]

    return tokens

token_to_id = {
    "[UNK]": 0,
    "h": 1,
    "p": 2,
    "##u": 3,
    "##g": 4,
    "##s": 5,
    "hu": 6,
    "hug": 7,
    "##gs": 8
}

next_id = max(token_to_id.values()) + 1

for token in vocab:

    if token not in token_to_id:

        token_to_id[token] = next_id

        next_id += 1

test_word = "hugs"

tokens = tokenize_word(
    test_word,
    set(token_to_id.keys())
)


print("\n--------------------------------")
print("FINAL TOKENIZATION")
print("--------------------------------")

print("Input Word:", test_word)

print("Tokens:", tokens)

ids = [
    token_to_id[token]
    for token in tokens
]

print("Token IDs:", ids)

unknown_word = "bum"

unknown_tokens = tokenize_word(
    unknown_word,
    set(token_to_id.keys())
)

print("\nUnknown Word:", unknown_word)

print("Tokens:", unknown_tokens)

Initial Splits:
hug -> ['h', '##u', '##g']
hugs -> ['h', '##u', '##g', '##s']
pug -> ['p', '##u', '##g']

Initial Vocabulary:
{'h', '##s', '##g', '##u', 'p'}

Step 1
Best Pair: ('h', '##u')
Score: 0.25
New Token: hu
Vocabulary: {'h', '##s', 'hu', '##g', '##u', 'p'}
Splits:
hug -> ['hu', '##g']
hugs -> ['hu', '##g', '##s']
pug -> ['p', '##u', '##g']

Step 2
Best Pair: ('p', '##u')
Score: 1.0
New Token: pu
Vocabulary: {'pu', 'h', '##s', 'hu', '##g', '##u', 'p'}
Splits:
hug -> ['hu', '##g']
hugs -> ['hu', '##g', '##s']
pug -> ['pu', '##g']

Step 3
Best Pair: ('hu', '##g')
Score: 0.25
New Token: hug
Vocabulary: {'pu', 'h', '##s', 'hu', '##g', 'hug', '##u', 'p'}
Splits:
hug -> ['hug']
hugs -> ['hug', '##s']
pug -> ['pu', '##g']

--------------------------------
FINAL TOKENIZATION
--------------------------------
Input Word: hugs
Tokens: ['hug', '##s']
Token IDs: [7, 5]

Unknown Word: bum
Tokens: ['[UNK]']
